# JRA-3Q Reanalysis: Seasonal Sea-Level Pressure

```{image} ../../thumbnails/gdex_logo.png
:alt: GDEX Cookbook logo
:width: 200px
```

---

## Overview

JRA-3Q is the Japan Meteorological Agency's third-generation global atmospheric
reanalysis, spanning 1947–present at 6-hourly resolution. GDEX hosts it as an ARCO dataset
[`d640000`](https://gdex.ucar.edu/datasets/d640000/).

Here we contrast winter and summer circulation by mapping the seasonal difference
(DJF − JJA) in **mean sea-level pressure** — opening the data directly from a
kerchunk reference file and using Dask to parallelize the remote reads.

1. Spin up a Dask `LocalCluster`
2. Open JRA-3Q sea-level pressure from a kerchunk reference
3. Compute DJF and JJA seasonal means over a climatology window
4. Map the DJF − JJA difference

## Prerequisites

| Concepts | Importance | Notes |
| --- | --- | --- |
| [Introduction to ARCO/Zarr on GDEX](../services/zarr_intro.ipynb) | Necessary | Opening kerchunk references on GDEX |
| [ERA5 Reanalysis Data Workflow](era5.ipynb) | Helpful | Same kerchunk access pattern |
| [xarray](https://docs.xarray.dev) | Necessary | Labeled N-D arrays |
| [Dask](https://docs.dask.org) | Necessary | Parallel / out-of-core compute |
| [Cartopy](https://scitools.org.uk/cartopy) | Necessary | Mapping |

- **Time to learn**: 25 minutes

---

## Imports

In [1]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from dask.distributed import LocalCluster

## Spin up a Dask cluster

The seasonal means read thousands of small 6-hourly fields over the network. A
`LocalCluster` fetches them in parallel and reduces them lazily, so memory stays
flat. Adjust `n_workers` and `memory_limit` (per worker) to suit your machine.

In [2]:
cluster = LocalCluster(
    n_workers=4,
    memory_limit="4GiB",)  # per worker
client = cluster.get_client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 16.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34441,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:36637,Total threads: 4
Dashboard: http://127.0.0.1:37347/status,Memory: 4.00 GiB
Nanny: tcp://127.0.0.1:33555,


## JRA-3Q on GDEX

We use **`prmsl-msl-an-ll125`** — pressure reduced to mean sea level (analysis), on
JRA-3Q's 1.25° latitude–longitude grid. GDEX serves the analysis surface fields as
a single kerchunk reference; we open it and pick out the pressure variable. You can
find references like this on the
[`d640000` data-access page](https://gdex.ucar.edu/datasets/d640000/dataaccess/)
under virtual reference files.

In [3]:
reference_url = "https://data.gdex.ucar.edu/d640000/kerchunk/anl_surf125-remote-osdf.parq"
reference_url

'https://data.gdex.ucar.edu/d640000/kerchunk/anl_surf125-remote-osdf.parq'

:::{note}
The `kerchunk` engine resolves the reference's chunk URLs; the array stays lazy
until we compute.
:::

In [4]:
ds = xr.open_dataset(reference_url, engine="kerchunk")
prmsl = ds["prmsl-msl-an-ll125"]
prmsl

FSTimeoutError: 

### Selecting two seasons

We bound the analysis to a short climatology window, then keep only the winter
(DJF) and summer (JJA) months — reading roughly half the window. Each seasonal mean collapses time to a single 2D field, and
we convert Pa to hPa.

:::{tip}
Widen the window (e.g. 1991–2020) for a more robust climatology if you have the
compute.
:::

In [ ]:
clim = prmsl.sel(time=slice("2015", "2019"))
month = clim.time.dt.month

djf = clim.sel(time=month.isin([12, 1, 2])).mean("time")
jja = clim.sel(time=month.isin([6, 7, 8])).mean("time")

diff = (djf - jja) / 100.0
diff.attrs["units"] = "hPa"

In [ ]:
%%time
diff = diff.compute()

## Mapping the seasonal contrast

Red marks where winter pressure exceeds summer — most strikingly the Siberian High
over Asia — while blue marks where it is lower, including the wintertime Aleutian
and Icelandic lows over the North Pacific and North Atlantic.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})

diff.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap="RdBu_r",
    center=0,
    robust=True,
    cbar_kwargs={"label": "DJF − JJA MSLP (hPa)", "shrink": 0.7},
)

ax.coastlines(color="black", linewidth=0.8)
ax.set_title("JRA-3Q Mean Sea-Level Pressure: DJF − JJA (2015–2019)")
plt.show()

### Interpreting the map

The difference field highlights the planet's main seasonal pressure centers:

- **Red over Asia** — the strongest signal on the map. Winter brings the cold,
  dense **Siberian High**, while summer brings the monsoon thermal low, so DJF
  pressure far exceeds JJA here.
- **Blue over the North Pacific and North Atlantic** — the wintertime **Aleutian**
  and **Icelandic lows**, which deepen in DJF and weaken in summer.
- **Deep blue around Antarctica** — the Southern Ocean storm belt, where pressure
  is lowest in the austral winter (JJA).

Together these capture the seasonal migration of the semi-permanent pressure
systems that drive mid-latitude weather.

In [ ]:
client.close()
cluster.close()

---

## Summary

We mapped JRA-3Q's winter-minus-summer mean sea-level pressure from a kerchunk
reference, using Dask to parallelize the reads. Selecting only DJF and JJA over a
bounded window kept the computation light enough to run interactively.

### What's next?

Continue to the [CESM2 Large Ensemble workflow](cesm2_lens.ipynb), which applies a
similar GDEX access pattern to a large climate-model ensemble.

## Resources and references

- [JRA-3Q overview (JMA)](https://jra.kishou.go.jp/JRA-3Q/index_en.html)
- [GDEX JRA-3Q dataset (`d640000`)](https://gdex.ucar.edu/datasets/d640000/)
- [kerchunk documentation](https://fsspec.github.io/kerchunk/)